# SupportGen · 06 — Model comparison (A / B / optional C, D) and export
1. Side-by-side comparison on the same test tickets:
   **A** base FLAN-T5-small · **B** our fine-tuned FLAN-T5-small · **C** Gemini *(optional)* ·
   **D** fine-tuned FLAN-T5-base *(only if you trained it)*
2. Export everything the web application needs as **one ZIP** → download → unzip on the Mac.

Gemini is only a **benchmark** — the project works without it. To enable it, add a Colab secret
named `GEMINI_API_KEY` (🔑 icon in the left sidebar → *Add new secret* → enable notebook access).
If the secret is missing, the Gemini column is skipped.

**Runtime:** GPU recommended (only needed if Model D outputs must be generated here).

## 0. Setup — Google Drive & project folder
Every SupportGen notebook starts with this same cell. It mounts Google Drive so that all
notebooks share one project folder (`MyDrive/SupportGen`): notebook 01 writes the cleaned
splits there, notebooks 02–06 read them. Nothing ever needs to be uploaded from your laptop.

Set `USE_DRIVE = False` only for a throw-away run — files then live in `/content` and are lost
when the Colab runtime disconnects.

In [1]:
# ── Setup (identical in every SupportGen notebook) ──────────────────────────
USE_DRIVE = True   # False → keep everything in /content (deleted when the runtime ends)

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/SupportGen"
else:
    BASE = "/content/SupportGen"
os.makedirs(BASE, exist_ok=True)
os.environ["SUPPORTGEN_BASE"] = BASE   # read by supportgen_common.py
sys.path.insert(0, BASE)
print("Project folder:", BASE)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/SupportGen


In [2]:
# Colab already ships torch, transformers, accelerate, scikit-learn, pandas, matplotlib.
# These are the extras SupportGen needs (already-installed packages are skipped).
!pip -q install rouge_score bert_score sacrebleu sentencepiece sentence-transformers google-genai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 3.2 MB/s eta 0:00:00


In [3]:
# Shared helpers were written to Drive by 01_dataset_preparation.ipynb
if not os.path.exists(os.path.join(BASE, "supportgen_common.py")):
    raise FileNotFoundError(
        f"supportgen_common.py not found in {BASE}. Run 01_dataset_preparation.ipynb first "
        "(with the same USE_DRIVE setting).")
import importlib, supportgen_common as sg
importlib.reload(sg)
sg.set_seed()
print("Device:", sg.get_device())

Device: cpu


In [4]:
import json, time, textwrap, shutil
import numpy as np, pandas as pd
from IPython.display import HTML

N_COMPARE = 100       # Gemini calls cost time / quota; all systems are scored on these same tickets
eval_df = sg.load_eval_subset()
cmp_df = eval_df.head(N_COMPARE).reset_index(drop=True)
prompts = [sg.build_prompt(s, b) for s, b in zip(cmp_df.subject, cmp_df.body)]

systems = {}
for run, label in [(sg.RUN_BASE, "A: base FLAN-T5-small"), (sg.RUN_FT, "B: fine-tuned FLAN-T5-small"),
                   (sg.RUN_FT_BASE, "D: fine-tuned FLAN-T5-base")]:
    if os.path.exists(sg.outputs_path(run)):
        o = pd.read_csv(sg.outputs_path(run), keep_default_na=False).set_index("uid").loc[cmp_df.uid]
        systems[label] = o.prediction.tolist()
missing = {"A: base FLAN-T5-small", "B: fine-tuned FLAN-T5-small"} - systems.keys()
if missing:
    raise FileNotFoundError(f"Missing outputs for {missing} — run notebooks 02–04 first.")
print("Loaded outputs for:", list(systems))

Loaded outputs for: ['A: base FLAN-T5-small', 'B: fine-tuned FLAN-T5-small']


### Optional: Model D outputs
If you fine-tuned FLAN-T5-base in notebook 03 but did not evaluate it in notebook 04, generate its
outputs for the comparison tickets here.

In [5]:
if "D: fine-tuned FLAN-T5-base" not in systems and os.path.exists(os.path.join(sg.FINETUNED_BASE_DIR, "training_config.json")):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    cfgD = json.load(open(os.path.join(sg.FINETUNED_BASE_DIR, "training_config.json")))
    tokD = AutoTokenizer.from_pretrained(sg.FINETUNED_BASE_DIR)
    mD = AutoModelForSeq2SeqLM.from_pretrained(sg.FINETUNED_BASE_DIR).to(sg.get_device())
    systems["D: fine-tuned FLAN-T5-base"], _ = sg.generate_batch(mD, tokD, prompts, cfgD["max_input_len"], cfgD["max_target_len"])
else:
    print("Model D:", "already loaded" if "D: fine-tuned FLAN-T5-base" in systems else "not trained — skipped")

Model D: not trained — skipped


## 1. Optional Gemini benchmark (Model C)
* Gemini receives the **same instruction prompt** as our models (zero-shot, no fine-tuning).
* `temperature=0` for (near-)deterministic output.
* Latency and token usage are measured per request. Cost is computed **only** if you fill in the current
  prices from the official Gemini pricing page — we do not guess prices.
* Model names change over time: set `GEMINI_MODEL` to a current Flash model from the Gemini docs.

In [6]:
GEMINI_MODEL = "gemini-2.5-flash"
PRICE_PER_1M_INPUT_USD = None      # e.g. fill in from https://ai.google.dev/pricing
PRICE_PER_1M_OUTPUT_USD = None
REQUEST_INTERVAL_S = 4.5           # stay under free-tier rate limits

try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception as e:
    GEMINI_KEY = None
    print("Gemini not configured (", type(e).__name__, ") — skipping Model C.")

gemini_stats = None
if GEMINI_KEY:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_KEY)
    g_out, g_lat, g_in_tok, g_out_tok, g_err = [], [], [], [], 0
    for i, p in enumerate(prompts):
        t0 = time.perf_counter()
        try:
            r = client.models.generate_content(model=GEMINI_MODEL, contents=p,
                                               config=types.GenerateContentConfig(temperature=0, max_output_tokens=1024))
            g_out.append((r.text or "").strip())
            u = r.usage_metadata
            g_in_tok.append(u.prompt_token_count or 0); g_out_tok.append(u.candidates_token_count or 0)
        except Exception as e:
            g_err += 1; g_out.append(""); print(f"request {i} failed: {type(e).__name__}: {str(e)[:120]}")
        g_lat.append(time.perf_counter() - t0)
        time.sleep(REQUEST_INTERVAL_S)
    systems["C: Gemini (" + GEMINI_MODEL + ")"] = g_out
    gemini_stats = {"model": GEMINI_MODEL, "requests": len(prompts), "errors": g_err,
                    "mean_latency_s": float(np.mean(g_lat)), "p95_latency_s": float(np.percentile(g_lat, 95)),
                    "total_input_tokens": int(sum(g_in_tok)), "total_output_tokens": int(sum(g_out_tok))}
    if PRICE_PER_1M_INPUT_USD is not None and PRICE_PER_1M_OUTPUT_USD is not None:
        cost = sum(g_in_tok) / 1e6 * PRICE_PER_1M_INPUT_USD + sum(g_out_tok) / 1e6 * PRICE_PER_1M_OUTPUT_USD
        gemini_stats |= {"estimated_cost_usd": cost, "cost_per_ticket_usd": cost / len(prompts)}
    pd.DataFrame({"uid": cmp_df.uid, "prediction": g_out, "latency_s": g_lat}).to_csv(sg.outputs_path(sg.RUN_GEMINI), index=False)
    display(pd.Series(gemini_stats))

Gemini not configured ( SecretNotFoundError ) — skipping Model C.


## 2. Metrics for every system on the same tickets
Note: Gemini was never trained on this dataset's answer style, so reference-based metrics (ROUGE,
BLEU) can penalise it even when its reply is good — this is exactly why the human evaluation matters.

In [7]:
refs = cmp_df.answer.tolist()
comparison = {name: sg.compute_gen_metrics(preds, refs) for name, preds in systems.items()}
cmp_table = pd.DataFrame(comparison)
display(cmp_table.loc[["rouge1", "rouge2", "rougeL", "bleu", "bertscore_f1", "bertscore_f1_rescaled",
                       "semantic_cosine", "avg_pred_words", "distinct_2", "empty_predictions"]])

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (20 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,A: base FLAN-T5-small,B: fine-tuned FLAN-T5-small
rouge1,0.223312,0.483491
rouge2,0.088896,0.244502
rougeL,0.176915,0.35109
bleu,1.280751,20.149911
bertscore_f1,0.869148,0.900125
bertscore_f1_rescaled,0.224692,0.408232
semantic_cosine,0.612932,0.787146
avg_pred_words,19.44,65.99
distinct_2,0.769523,0.298969
empty_predictions,0,0


## 3. Side-by-side examples

In [8]:
side = pd.DataFrame({"ticket": "<b>" + cmp_df.subject + "</b><br>" + cmp_df.body.str.slice(0, 500), "reference": refs}
                    | {k: v for k, v in systems.items()})
side.to_csv(os.path.join(sg.RESULTS_DIR, "comparison_examples.csv"), index=False)
HTML(side.head(8).to_html(escape=False, index=False).replace("<td>", '<td style="text-align:left;vertical-align:top;max-width:320px">'))

ticket,reference,A: base FLAN-T5-small,B: fine-tuned FLAN-T5-small
Ensuring the Safety of Medical DataRequesting advice on securing medical data using Microsoft OneNote and Kaspersky Internet Security. Would like to know if these tools can protect sensitive medical information from unauthorized access. Could you provide guidance on setting up encryption and access controls to ensure the confidentiality and integrity of the medical data? Appreciate your assistance.,"Thank you for your concern about securing medical data. I am happy to provide guidance on the best practices for using Microsoft OneNote and Kaspersky Internet Security. To protect sensitive medical information from unauthorized access, I recommend setting up encryption and access controls. In Microsoft OneNote, you can use password protection and encryption to secure your notebooks. Additionally, Kaspersky Internet Security offers an extra layer of protection against malware and unauthorized access. I will provide detailed information on how to configure these settings.",Thanks for your help. I've been able to use oneNote and Kaspersky Internet Security for a few years and I'm glad I did.,"We can provide guidance on securing medical data using Microsoft OneNote and Kaspersky Internet Security. To ensure the confidentiality and integrity of the medical data, we recommend setting up encryption and access controls. Please let us know a convenient time for a call at tel_num> to discuss further."
"Hello Customer Support, I am encountering issues with integrating data among my analytics tools, which is impacting my investment optimization. The problem might be due to compatibility issues or outdated software. So far, I have checked that Node.js is up to date and verified the API connections in Evernote and Salesforce CRM, but the issue still persists. My team is working on resolving this issue, and we would greatly appreciate any guidance you can provide to help troubleshoot and resolve it",Please contact +1-800-123-4567 to discuss the steps needed to resolve the data integration issue.,Node.js is up to date and verified the API connections in Evernote and Salesforce CRM,"Hello [Name], thank you for reaching out to us regarding the integration issues with your analytics tools. We are sorry to hear that you are experiencing issues with Node.js and Salesforce CRM. To better assist you, could you please provide us with more details about the error messages you are receiving and the steps you have taken so far? We would like to schedule a call at [Tel Num] to discuss this further. Please let us know a convenient time for you. We look forward to resolving this issue as soon as possible."
"Discrepancies Detected in Investment ForecastsCustomer Support is reporting a problem with the financial firm's investment forecasts. They observed that the predictions are inaccurate and suspect the issue may stem from faulty data integration. Despite efforts by the team to recalibrate algorithms and data sources, the problem remains unresolved. They are concerned that this could impact investment decisions and are requesting assistance to resolve the issue. Could you please review and suggest possible solutions or improvements? Your promp","We received your email regarding discrepancies in investment forecasts, likely due to flawed data integration. We are currently investigating the issue and will provide recommendations for improvements. Kindly share details about the algorithms and data sources involved, and contact us at for further discussion.",Subject: Discrepancies in Investment Forecasts,"name>, thank you for reporting the issue with the investment forecasts. We apologize for the inconvenience caused by the inaccurate predictions. To better assist you, could you please provide details on the specific data sources and data sources involved? We would like to schedule a call at tel_num> to discuss and explore possible solutions or improvements. Please let us know a convenient time for

In [9]:
sg.save_json({"n_tickets": len(cmp_df), "metrics": comparison, "gemini": gemini_stats,
              "deployment_notes": {
                  "fine_tuned_flan_t5_small": "runs locally (CPU/MPS/GPU), no API key, no per-request cost, data stays on-premise",
                  "gemini": "requires network + API key, per-token cost, data sent to a third party"}},
             os.path.join(sg.RESULTS_DIR, "comparison_metrics.json"))
if gemini_stats:
    m = comparison["C: Gemini (" + GEMINI_MODEL + ")"]
    sg.log_experiment(sg.RUN_GEMINI, model=GEMINI_MODEL, rouge1=m["rouge1"], rouge2=m["rouge2"], rougeL=m["rougeL"],
                      bleu=m["bleu"], bertscore_f1=m["bertscore_f1"], semantic_cosine=m["semantic_cosine"],
                      notes=f"Zero-shot API benchmark on {len(cmp_df)} tickets")
print("saved comparison_metrics.json")

saved comparison_metrics.json


## 4. Export for the web application
The ZIP has this layout, so unzipping it **inside the project folder** puts everything where the
Django backend expects it:
```
models/flan-t5-supportgen/   ← fine-tuned model + tokenizer + training_config.json
models/classifiers/          ← queue_group / priority / type pipelines + meta.json
models/flan-t5-base-supportgen/  (only if INCLUDE_MODEL_D)
results/                     ← metrics JSON/CSV + plots (shown on the Evaluation page)
results/data/                ← config.json, eda_stats.json, split_sizes.json
```
Checkpoints and the dataset splits are **not** included (not needed for inference).

In [10]:
INCLUDE_MODEL_D = False
stage = "/content/supportgen_export"
shutil.rmtree(stage, ignore_errors=True)

def copy_dir(src, dst):
    if os.path.isdir(src):
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("checkpoint-*", "*.pt", "optimizer*", "rng_state*"))
        print("✓", os.path.relpath(dst, stage))
    else:
        print("✗ missing", src)

copy_dir(sg.FINETUNED_DIR, f"{stage}/models/flan-t5-supportgen")
copy_dir(sg.CLASSIFIER_DIR, f"{stage}/models/classifiers")
if INCLUDE_MODEL_D:
    copy_dir(sg.FINETUNED_BASE_DIR, f"{stage}/models/flan-t5-base-supportgen")
copy_dir(sg.RESULTS_DIR, f"{stage}/results")
os.makedirs(f"{stage}/results/data", exist_ok=True)
for f in ["config.json", "eda_stats.json", "split_sizes.json"]:
    if os.path.exists(os.path.join(sg.DATA_DIR, f)):
        shutil.copy(os.path.join(sg.DATA_DIR, f), f"{stage}/results/data/{f}")
if os.path.exists(sg.EXPERIMENTS_CSV):
    shutil.copy(sg.EXPERIMENTS_CSV, f"{stage}/results/experiments.csv")

zip_path = shutil.make_archive("/content/supportgen_models", "zip", root_dir=stage)
shutil.copy(zip_path, os.path.join(sg.EXPORTS_DIR, "supportgen_models.zip"))
print(f"\nZIP: {zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB) — also copied to {sg.EXPORTS_DIR}")

✓ models/flan-t5-supportgen
✓ models/classifiers
✓ results

ZIP: /content/supportgen_models.zip (313 MB) — also copied to /content/drive/MyDrive/SupportGen/exports


In [11]:
# Download to your computer (or grab it later from Drive: MyDrive/SupportGen/exports/)
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. On your Mac
```bash
cd "<path-to>/project"
unzip -o ~/Downloads/supportgen_models.zip -d .
ls models/flan-t5-supportgen      # config.json, model.safetensors, tokenizer files, training_config.json
```
The Django backend reads the model from `SUPPORTGEN_MODEL_DIR` (default `models/flan-t5-supportgen`)
and the classifiers from `SUPPORTGEN_CLASSIFIER_DIR` (default `models/classifiers`) — see `.env.example`.

Install the same scikit-learn version as recorded in `models/classifiers/meta.json` in the backend
environment, so the `.joblib` files load correctly.